(course-core-14)=

# Module 14: System Comparison and Validation

**Welcome back, Apprentice Master.** In [Module 13: Topological Analysis](../Common_Core/13_Topological_Analysis.ipynb), you explored non-spatial topological features using `msm.topology`. Now we address automated quality assurance: **System Comparison and Validation** using **`msm.contains()`**, **`msm.is_composed_of()`**, and **`msm.compare()`**.

When building computational pipelines—such as converting file formats, extracting sub-structures, or preparing simulation boxes—verifying system composition, attribute presence, and topological equality is essential. Did a file conversion drop water molecules? Does a system contain hydrogen atoms or unit cell box vectors? Are an extracted sub-system and its parent selection identical?

MolSysMT provides three form-agnostic validation functions in `msm.basic`:
- **`msm.contains()`**: Checks for **presence**—whether specific elements, attributes, or molecule types exist in a system.
- **`msm.is_composed_of()`**: Checks for **exclusivity and quantitative composition**—whether a system is made up *solely* of specified components or exact element counts.
- **`msm.compare()`**: Compares two molecular systems across different data forms to audit attribute equality.

(course-core-14-learning-outcomes)=
```{admonition} Learning Outcomes
:class: dropdown learning-outcomes

By the end of this module, you will be able to:
- Audit attribute, element, and molecule presence using `msm.contains()`.
- Understand the key distinction between presence (`contains`) and exclusive composition (`is_composed_of`).
- Validate quantitative system composition using `msm.is_composed_of()`.
- Compare systems across different data forms using `msm.compare()`.
- Perform targeted cross-selection comparisons using `selection` and `selection_2`.
- Generate detailed dictionary reports of attribute mismatches (`output_type='dictionary'`).
```

### 1. Checking Attribute Presence

Let's begin by importing MolSysMT and loading our T4 Lysozyme demonstration system.

In [1]:
import molsysmt as msm
from molsysmt import systems

# Load T4 Lysozyme file
lysozyme_file = systems['T4 lysozyme L99A']['181l.bcif.gz']

# Convert to native MolSys object
lysozyme_molsys = msm.convert(lysozyme_file, to_form='molsysmt.MolSys')

The function **`msm.contains()`** checks whether a molecular system (or selection) contains specific molecule types, structural elements, or physical attributes. It returns `True` as long as the requested item is present, regardless of what else the system contains:

In [2]:
# Check if the system contains proteins
has_protein = msm.contains(lysozyme_molsys, proteins=True)
print(f"Does the system contain proteins? {has_protein}")

# Check if the system contains water molecules and ions
has_water_and_ions = msm.contains(lysozyme_molsys, waters=True, ions=True)
print(f"Does the system contain water and ions? {has_water_and_ions}")

# Check if the system contains periodic boundary box vectors
has_box = msm.contains(lysozyme_molsys, box=True)
print(f"Does the system contain periodic box vectors? {has_box}")

# Check for absence of lipids
no_lipids = msm.contains(lysozyme_molsys, lipids=False)
print(f"Is the system free of lipids? {no_lipids}")

Does the system contain proteins? True
Does the system contain water and ions? True
Does the system contain periodic box vectors? True
Is the system free of lipids? True


:::{hint}
:class: dropdown
**msm.contains()**: Form-agnostic function to check whether a molecular system or selection contains specified elements, molecule types, or attributes. See API doc: {func}`molsysmt.basic.contains`.
:::

### 2. Verifying System Composition

While `msm.contains()` asks *"is this item present?"*, **`msm.is_composed_of()`** asks *"is the system made up exclusively of these items?"*.

Because our T4 Lysozyme system contains a protein **plus** water molecules, ions, and ligands, checking if it is composed *only* of proteins returns `False`:

In [3]:
# Test for protein exclusivity (returns False because waters/ions exist)
is_protein_only = msm.is_composed_of(lysozyme_molsys, proteins=True)
print(f"Is the system composed ONLY of proteins? {is_protein_only}")

# Test full composition matching all component types (returns True)
is_full_composition = msm.is_composed_of(lysozyme_molsys, proteins=True, waters=True, small_molecules=True, ions=True)
print(f"Does the system match full exclusive composition? {is_full_composition}")

Is the system composed ONLY of proteins? False
Does the system match full exclusive composition? True


You can also use `msm.is_composed_of()` to validate exact quantitative counts of structural elements or specific selections:

In [4]:
# Verify exact number of structural chains
is_6_chains = msm.is_composed_of(lysozyme_molsys, n_chains=6)
print(f"Is the system composed of exactly 6 chains? {is_6_chains}")

# Verify that the protein selection is composed of exactly 1 protein molecule
is_1_protein = msm.is_composed_of(lysozyme_molsys, selection='molecule_type == "protein"', proteins=1)
print(f"Is the protein selection composed of exactly 1 protein molecule? {is_1_protein}")

Is the system composed of exactly 6 chains? True
Is the protein selection composed of exactly 1 protein molecule? True


:::{hint}
:class: dropdown
**msm.is_composed_of()**: Checks whether a molecular system or selection matches exact quantitative and exclusive composition criteria. See API doc: {func}`molsysmt.basic.is_composed_of`.
:::

### 3. Comparing Systems Across Forms

When converting file formats or processing trajectory frames, **`msm.compare()`** validates whether two molecular systems have identical core attributes, regardless of their data forms:

In [5]:
# Compare raw file directly against in-memory MolSys object
is_identical = msm.compare(lysozyme_file, lysozyme_molsys)
print(f"Are the file and in-memory object identical? {is_identical}")

Are the file and in-memory object identical? True


When comparing non-identical systems (such as a full system versus an extracted protein sub-system), set `output_type='dictionary'` to obtain a per-attribute breakdown of matching and mismatching structural counts:

In [6]:
# Extract protein-only sub-system
protein_only = msm.extract(lysozyme_molsys, selection='molecule_type == "protein"')

# Audit count attribute differences with output_type='dictionary'
report = msm.compare(lysozyme_molsys, protein_only, n_atoms=True, n_groups=True, n_chains=True, box=True, output_type='dictionary')

print("Attribute comparison audit (Full vs Protein-Only):")
for attr, matches in report.items():
    status = "MATCH" if matches else "MISMATCH"
    print(f"- {attr}: {status}")

Attribute comparison audit (Full vs Protein-Only):
- n_atoms: MISMATCH
- n_groups: MISMATCH
- n_chains: MISMATCH
- box: MATCH


Finally, you can perform targeted cross-selection comparisons between two systems using `selection` and `selection_2`:

In [7]:
# Compare protein subset of full system against all of protein_only system
selection_match = msm.compare(lysozyme_molsys, protein_only, 
                              selection='molecule_type == "protein"', selection_2='all',
                              coordinates=True, n_groups=True, output_type='dictionary')
print(f"Targeted cross-selection comparison: {selection_match}")

Targeted cross-selection comparison: {'n_groups': True, 'coordinates': True}


:::{hint}
:class: dropdown
**msm.compare()**: Universal comparison engine for checking attribute equality across molecular systems. See API doc: {func}`molsysmt.basic.compare`.
:::

--- 

### 🏆 Challenge 14: The System Validator

1. Load the **T4 Lysozyme** system (`systems['T4 lysozyme L99A']['181l.bcif.gz']`).
2. Use `msm.contains()` to check if the system contains water molecules and periodic box vectors.
3. Use `msm.is_composed_of()` to demonstrate that `proteins=True` returns `False`, but adding `waters=True, small_molecules=True, ions=True` returns `True`.
4. Extract chain A into `chain_a` and chain B into `chain_b`, and use `msm.compare(chain_a, chain_b, output_type='dictionary')` to audit differences.

System validation ensures reproducible computational pipelines. In [Module 15: Semantic Labeling](../Common_Core/15_Semantic_Labeling.ipynb), we will explore chemical labeling and secondary structure annotations.

```{key-takeaway}
MolSysMT provides a complete validation toolkit: `msm.contains()` checks attribute presence, `msm.is_composed_of()` verifies exact exclusive composition, and `msm.compare()` performs form-agnostic attribute comparisons across systems.
```

(course-core-14-see-also)=
:::{seealso}
:class: dropdown
**API Documentation for Functions in this Module:**
- {func}`molsysmt.basic.contains` — Attribute and element presence checker.
- {func}`molsysmt.basic.is_composed_of` — Quantitative composition verifier.
- {func}`molsysmt.basic.compare` — Universal system comparison engine.

**Related Course Modules & Guides:**
- Previous Module: [Module 13: Topological Analysis](../Common_Core/13_Topological_Analysis.ipynb)
- Next Module: [Module 15: Semantic Labeling](../Common_Core/15_Semantic_Labeling.ipynb)
- User Guide: {ref}`user-foundations`
:::